# Point-in-Time Universe Construction — Big Caps and NYSE P20–P50

This notebook produces exactly two daily datasets. Size breakpoints are estimated from the NYSE snapshot observable at the close of formation month $M$. The NYSE P90 breakpoint is then applied to eligible NYSE, AMEX, and NASDAQ securities for the big-cap Top 100. The P20–P50 Top 100 remains restricted to NYSE securities, as specified in the article.

Every selected security must have 60 strictly consecutive and complete months of daily returns through $M$. Formation membership is recorded on the market-close row of $M$ and is therefore independent of whether the security remains observable in $M+1$. No return is imputed.

In [ ]:
# 1. CONFIGURATION
from pathlib import Path
import polars as pl

ROOT = Path.cwd()
if ROOT.name in {'notebooks', 'tests'}:
    ROOT = ROOT.parent

SRC = ROOT / 'data' / 'processed' / 'crsp_daily_common_stock_pit_source-1990-2025.parquet'
OUT_BIG = ROOT / 'data' / 'processed' / 'nyse_big_caps_pit_daily.parquet'
OUT_SMALL = ROOT / 'data' / 'processed' / 'nyse_small_caps_p20_p50_pit_daily.parquet'

INVESTABLE_EXCHANGES = ['N', 'A', 'Q']  # NYSE, AMEX, NASDAQ
NYSE_CODE = 'N'
W_MAX = 60
TOP_N = 100
P20, P50, P90 = 0.20, 0.50, 0.90
WRITE_OUTPUTS = True

assert SRC.exists(), SRC
print(f'Source: {SRC} ({SRC.stat().st_size / 2**30:.2f} GiB)')
print(f'Polars: {pl.__version__}')
print(f'Investable formation exchanges: {INVESTABLE_EXCHANGES}')
print(f'Maximum common history: {W_MAX} months')

## 2. Source contract and investable daily history

Security eligibility and return observation are deliberately separated. The canonical source produced by notebook 00 identifies ordinary common stocks point in time and combines this classification with regular/active status and the three admitted exchanges (NYSE, AMEX, and NASDAQ). This eligibility flag governs the estimation history and the formation snapshot. Once a security is selected, however, its complete CRSP return path is retained for the same `PERMNO`, irrespective of a subsequent status or exchange-code change. Thus a migration or delisting during $M+1$ neither creates ex-ante eligibility nor truncates the realised holding return.

In [ ]:
lf = pl.scan_parquet(SRC)
schema = lf.collect_schema()
required = {
    'PERMNO', 'Ticker', 'DlyCalDt', 'DlyRet', 'DlyRetx', 'DlyRetI',
    'DlyPrc', 'DlyCap', 'PrimaryExch', 'DelistingDt', 'DelRet',
    'DelReasonType', 'delist_category',
    'is_common_stock_10_11', 'is_regular_active',
    'is_investable_exchange', 'is_formation_eligible',
}
missing = required - set(schema.names())
assert not missing, f'Missing source columns: {sorted(missing)}'

# Complete return source: never truncate a selected PERMNO because its exchange code changes.
daily_source = (
    lf.select(sorted(required))
      .unique(subset=['PERMNO', 'DlyCalDt'], keep='first')
      .with_columns(pl.col('DlyCalDt').dt.truncate('1mo').alias('month'))
)

# Formation/history source: the canonical row-level PIT flag combines the
# ordinary-common-stock mapping, regular/active status, and N/A/Q eligibility.
# `daily_source` itself remains unfiltered so that M+1 and delisting paths survive.
daily_us = daily_source.filter(pl.col('is_formation_eligible'))
print('Source contract validated.')

## 3. Monthly snapshot and strict daily completeness

The reference calendar is the union of observed trading dates across NYSE, AMEX, and NASDAQ. A security-month is complete only when it contains one non-missing return on every reference trading date. The formation snapshot additionally requires observation at the common market close, positive market capitalisation, a valid price, and no delisting known by that close.

In [ ]:
market_month = (
    daily_us.select(['month', 'DlyCalDt']).unique()
    .group_by('month')
    .agg(
        pl.col('DlyCalDt').max().alias('market_end'),
        pl.col('DlyCalDt').n_unique().alias('n_market_days'),
    )
)

security_month = (
    daily_us.sort(['PERMNO', 'DlyCalDt'])
    .group_by(['PERMNO', 'month'])
    .agg(
        pl.col('Ticker').last().alias('Ticker'),
        pl.col('PrimaryExch').last().alias('formation_exchange'),
        pl.col('DlyCalDt').last().alias('security_last_date'),
        pl.col('DlyCap').last().alias('mktcap'),
        pl.col('DlyPrc').last().alias('month_end_price'),
        pl.col('DlyCalDt').n_unique().alias('n_security_days'),
        pl.col('DlyRet').is_not_null().sum().alias('n_valid_returns'),
        pl.col('DelistingDt').drop_nulls().last().alias('DelistingDt'),
    )
    .join(market_month, on='month', how='left')
    .with_columns(
        (pl.col('security_last_date') == pl.col('market_end')).alias('observed_at_market_end'),
        (
            pl.col('DelistingDt').is_not_null()
            & (pl.col('DelistingDt') <= pl.col('market_end'))
        ).alias('delisted_by_formation'),
    )
    .with_columns(
        (
            pl.col('observed_at_market_end')
            & ~pl.col('delisted_by_formation')
            & pl.col('mktcap').is_not_null()
            & (pl.col('mktcap') > 0)
            & pl.col('month_end_price').is_not_null()
        ).alias('alive_at_formation'),
        (
            (pl.col('n_security_days') == pl.col('n_market_days'))
            & (pl.col('n_valid_returns') == pl.col('n_market_days'))
        ).alias('complete_daily_month'),
    )
)

monthly = security_month.collect(engine='streaming')
print(f'Security-month observations: {len(monthly):,}')
display(monthly.select([
    pl.len().alias('rows'),
    pl.col('alive_at_formation').sum().alias('alive_rows'),
    pl.col('complete_daily_month').sum().alias('complete_months'),
]))

In [ ]:
# 4. STRICTLY CONSECUTIVE 60-MONTH HISTORY
monthly_hist = (
    monthly.lazy()
    .with_columns(
        (pl.col('month').dt.year() * 12 + pl.col('month').dt.month()).alias('month_id')
    )
    .sort(['PERMNO', 'month_id'])
    .with_columns(
        pl.col('month_id').shift(W_MAX - 1).over('PERMNO').alias('month_id_lag59'),
        pl.col('complete_daily_month').cast(pl.Int16)
          .rolling_sum(window_size=W_MAX, min_samples=W_MAX)
          .over('PERMNO').alias('n_complete_last60'),
    )
    .with_columns(
        (
            pl.col('month_id_lag59').is_not_null()
            & ((pl.col('month_id') - pl.col('month_id_lag59')) == W_MAX - 1)
            & (pl.col('n_complete_last60') == W_MAX)
        ).alias('strict_hist60')
    )
    .collect(engine='streaming')
)

assert monthly_hist.filter(pl.col('strict_hist60') & ~pl.col('complete_daily_month')).is_empty()
print(f'Strictly eligible security-month rows: {monthly_hist["strict_hist60"].sum():,}')

## 5. NYSE breakpoints and formation memberships

Breakpoints are computed before applying the 60-month history filter and from the alive NYSE snapshot only. The P90 threshold is applied to all eligible NYSE–AMEX–NASDAQ securities. The P20–P50 condition is applied only to eligible NYSE securities. Ranking is performed after these conditions, with market capitalisation descending and PERMNO as the deterministic tie-breaker.

In [ ]:
nyse_reference = monthly_hist.filter(
    pl.col('alive_at_formation') & (pl.col('formation_exchange') == NYSE_CODE)
)
breakpoints = (
    nyse_reference.group_by('month')
    .agg(
        pl.col('mktcap').quantile(P20).alias('bp_p20'),
        pl.col('mktcap').quantile(P50).alias('bp_p50'),
        pl.col('mktcap').quantile(P90).alias('bp_p90'),
        pl.len().alias('n_nyse_snapshot'),
    )
)

eligible = (
    monthly_hist.join(breakpoints, on='month', how='inner')
    .filter(pl.col('alive_at_formation'), pl.col('strict_hist60'))
)

def build_membership(bucket: str) -> pl.DataFrame:
    if bucket == 'big_caps':
        condition = pl.col('mktcap') > pl.col('bp_p90')
    elif bucket == 'small_caps_p20_p50':
        condition = (
            (pl.col('formation_exchange') == NYSE_CODE)
            & (pl.col('mktcap') > pl.col('bp_p20'))
            & (pl.col('mktcap') <= pl.col('bp_p50'))
        )
    else:
        raise ValueError(bucket)

    membership = (
        eligible.filter(condition)
        .sort(['month', 'mktcap', 'PERMNO'], descending=[False, True, False])
        .with_columns(
            pl.col('mktcap').rank(method='ordinal', descending=True).over('month')
              .cast(pl.Int16).alias('rank')
        )
        .filter(pl.col('rank') <= TOP_N)
        .with_columns(
            pl.col('month').alias('formation_month'),
            pl.col('month').dt.offset_by('1mo').alias('active_month'),
            pl.lit(bucket).alias('bucket'),
        )
        .select([
            'formation_month', 'active_month', 'market_end', 'PERMNO', 'Ticker',
            'formation_exchange', 'bucket', 'rank',
            pl.col('mktcap').alias('formation_mktcap'),
            'bp_p20', 'bp_p50', 'bp_p90', 'n_nyse_snapshot',
            'strict_hist60', 'security_last_date',
        ])
        .sort(['formation_month', 'rank'])
    )
    counts = membership.group_by('formation_month').agg(
        pl.len().alias('n'), pl.col('rank').n_unique().alias('n_ranks')
    )
    assert counts.filter((pl.col('n') != TOP_N) | (pl.col('n_ranks') != TOP_N)).is_empty(), (
        bucket, counts.filter((pl.col('n') != TOP_N) | (pl.col('n_ranks') != TOP_N))
    )
    return membership

membership_big = build_membership('big_caps')
membership_small = build_membership('small_caps_p20_p50')

def membership_summary(membership: pl.DataFrame, name: str):
    counts = membership.group_by('formation_month').agg(pl.len().alias('n')).sort('formation_month')
    exchanges = membership.group_by('formation_exchange').agg(pl.len().alias('memberships')).sort('formation_exchange')
    print(f'{name}: {len(membership):,} memberships | months={len(counts)} | n={counts["n"].min()}–{counts["n"].max()}')
    display(exchanges)

membership_summary(membership_big, 'Big Caps — NYSE breakpoint applied to N/A/Q')
membership_summary(membership_small, 'P20–P50 — NYSE only')

## 6. Daily exports with two distinct PIT flags

`is_formation_member` identifies the Top 100 known at the close of $M$ and is the only membership flag used by the signal engine. It is attached to the actual formation-close row, which always exists by construction. `is_active` indicates that the security has an observed daily row during the subsequent active month $M+1$ and is retained for portfolio-return diagnostics. The two concepts must not be conflated.

In [ ]:
def build_daily_export(membership: pl.DataFrame) -> pl.LazyFrame:
    selected_permnos = membership.lazy().select('PERMNO').unique()

    active_meta = membership.lazy().select([
        'PERMNO', 'active_month', 'formation_month', 'formation_exchange',
        'bucket', 'rank', 'formation_mktcap', 'bp_p20', 'bp_p50', 'bp_p90',
        'n_nyse_snapshot', 'strict_hist60',
    ])

    formation_meta = membership.lazy().select([
        'PERMNO',
        pl.col('market_end').alias('DlyCalDt'),
        pl.col('formation_month').alias('formation_member_month'),
        pl.col('active_month').alias('formation_member_active_month'),
        pl.col('formation_exchange').alias('formation_member_exchange'),
        pl.col('bucket').alias('formation_member_bucket'),
        pl.col('rank').alias('formation_member_rank'),
        pl.col('formation_mktcap').alias('formation_member_mktcap'),
        pl.col('bp_p20').alias('formation_member_bp_p20'),
        pl.col('bp_p50').alias('formation_member_bp_p50'),
        pl.col('bp_p90').alias('formation_member_bp_p90'),
        pl.col('n_nyse_snapshot').alias('formation_member_n_nyse_snapshot'),
        pl.col('strict_hist60').alias('formation_member_strict_hist60'),
    ])

    event = (
        pl.col('DelistingDt').is_not_null()
        & (pl.col('DlyCalDt').dt.date() == pl.col('DelistingDt').dt.date())
    )

    return (
        # Membership is formed on N/A/Q, but the realised path follows the same
        # PERMNO across any later exchange migration during the holding month.
        daily_source.join(selected_permnos, on='PERMNO', how='inner')
        .rename({'month': 'active_month'})
        .join(active_meta, on=['PERMNO', 'active_month'], how='left')
        .join(formation_meta, on=['PERMNO', 'DlyCalDt'], how='left')
        .with_columns(
            pl.col('rank').is_not_null().cast(pl.Int8).alias('is_active'),
            pl.col('formation_member_rank').is_not_null().cast(pl.Int8).alias('is_formation_member'),
            pl.when(event).then(pl.col('DelRet')).otherwise(None).alias('DelRet_event'),
            pl.when(event).then(pl.col('DelReasonType')).otherwise(None).alias('DelReasonType_event'),
            pl.when(event).then(pl.col('delist_category')).otherwise(None).alias('delist_category_event'),
            event.alias('is_delisting_event'),
        )
        .select([
            'PERMNO', 'Ticker', 'PrimaryExch', 'DlyCalDt', 'DlyRet', 'DlyRetx',
            'DlyRetI', 'DlyPrc', 'DlyCap',
            'is_common_stock_10_11', 'is_regular_active',
            'is_investable_exchange', 'is_formation_eligible',
            'active_month', 'formation_month', 'is_active', 'formation_exchange',
            'bucket', 'rank', 'formation_mktcap', 'bp_p20', 'bp_p50', 'bp_p90',
            'n_nyse_snapshot', 'strict_hist60',
            'formation_member_month', 'formation_member_active_month',
            'is_formation_member', 'formation_member_exchange',
            'formation_member_bucket', 'formation_member_rank',
            'formation_member_mktcap', 'formation_member_bp_p20',
            'formation_member_bp_p50', 'formation_member_bp_p90',
            'formation_member_n_nyse_snapshot',
            'formation_member_strict_hist60',
            'is_delisting_event', 'DelRet_event', 'DelReasonType_event',
            'delist_category_event',
        ])
        .sort(['PERMNO', 'DlyCalDt'])
    )

export_big = build_daily_export(membership_big)
export_small = build_daily_export(membership_small)
print('Lazy export plans constructed; no file has been written yet.')

In [ ]:
# 7. ATOMIC REPLACEMENT OF THE TWO DATASETS
def sink_atomic(frame: pl.LazyFrame, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + '.tmp')
    if temporary.exists():
        temporary.unlink()
    frame.sink_parquet(temporary, compression='zstd')
    temporary.replace(path)

if WRITE_OUTPUTS:
    print('Writing big-cap dataset...')
    sink_atomic(export_big, OUT_BIG)
    print('Writing NYSE P20–P50 dataset...')
    sink_atomic(export_small, OUT_SMALL)
    print('Written:')
    print(' -', OUT_BIG)
    print(' -', OUT_SMALL)
else:
    print('WRITE_OUTPUTS=False — no file written.')

## 8. Output summary

The decisive count is `formation_memberships`: it must equal 100 at every formation month. `active_daily_rows` may reflect later disappearance and must never be used to reconstruct formation membership.

In [ ]:
for label, path in [('Big Caps', OUT_BIG), ('NYSE P20–P50', OUT_SMALL)]:
    if not path.exists():
        print(label, ': file absent')
        continue
    frame = pl.scan_parquet(path)
    summary = frame.select(
        pl.len().alias('rows'),
        pl.col('PERMNO').n_unique().alias('permnos_union'),
        pl.col('DlyCalDt').min().alias('date_min'),
        pl.col('DlyCalDt').max().alias('date_max'),
        pl.col('is_formation_member').sum().alias('formation_memberships'),
        pl.col('is_active').sum().alias('active_daily_rows'),
    ).collect(engine='streaming')
    counts = (
        frame.filter(pl.col('is_formation_member') == 1)
        .group_by('formation_member_month').agg(pl.len().alias('n'))
        .select(pl.col('n').min().alias('n_min'), pl.col('n').max().alias('n_max'))
        .collect(engine='streaming')
    )
    print(label)
    display(summary)
    display(counts)

print('Next: run tests/01_assert_pit_big_small_caps.ipynb.')